# Visualize static PDBs from protein databank

<video controls src="./assets/pdb_visualization.webm">

Setup runner with simulations:

In [ ]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(port=0, name="EXAMPLE: Visualise static PDBs from databank")

Fetch PDBs and convert to MDAnalysis universes

In [ ]:
from fetch_pdbs import fetch_pdb_universes

universes = fetch_pdb_universes("1AKE", "4BWZ", "3M1N", to_guess=("bonds", "types", "masses"))

Convert to NanoVer simulations, add to runner with cartoon rendering selection:

In [ ]:
from nanover.app import RenderingSelection, SELECTION_ROOT_ID
from nanover.mdanalysis import UniverseSimulation

simulations = [UniverseSimulation.from_universe(universe, name=name) for name, universe in universes.items()]
cartoon = RenderingSelection(SELECTION_ROOT_ID)
cartoon.renderer = "cartoon"

for name, universe in universes.items():
    simulation = UniverseSimulation.from_universe(universe, name=name)
    imd_runner.add_simulation(simulation)
    imd_runner.set_simulation_selections(simulation, cartoon)

Use Nanover jupyter utilities to make convenience commands for switching between simulations:

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [ ]:
def make_switch(i, simulation):
    def switch():
        imd_runner.load(i)
        utilities.notify_all(f"Loading {simulation.name}")
    return switch

for i, simulation in enumerate(simulations):
    imd_runner.app_server.register_command(f"user/sim-{i}", make_switch(i, simulation), label=simulation.name, icon="🫧")